# Human Activity Recognition using Hidden Markov Models

This notebook implements a data pipeline for collecting, preprocessing, and extracting features from smartphone sensor data (accelerometer and gyroscope) to classify four human activities: standing, walking, jumping, and still.

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from scipy.fft import fft, fftfreq
from sklearn.preprocessing import StandardScaler
import json
import logging

os.makedirs('logs', exist_ok=True)
logging.basicConfig(filename='logs/pipeline.log', level=logging.INFO,
                    format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

## 1. Data Loading

Load all sensor recordings from both team members. Each zip contains Accelerometer.csv, Gyroscope.csv, Metadata.csv, and Tags.csv.

- David: iPhone 14, 100Hz sampling rate (10ms interval)
- Queen: iPhone SE (2nd gen), 100Hz sampling rate (10ms interval)

Both phones were set to the same 100Hz sampling rate in Sensor Logger, so no resampling or interpolation was needed to harmonize the data. The timestamps and sample counts are directly comparable across both devices.

In [ ]:
def get_activity_label(zip_path, default_label=None):
    with zipfile.ZipFile(zip_path) as z:
        if 'Tags.csv' in z.namelist():
            tags = z.read('Tags.csv').decode().strip().split('\n')
            if len(tags) > 1:
                tag = tags[1].strip().lower()
                if 'standing' in tag:
                    return 'standing'
                elif 'walking' in tag:
                    return 'walking'
                elif 'jumping' in tag:
                    return 'jumping'
                elif 'still' in tag or 'rest' in tag:
                    return 'still'
        
        name = os.path.basename(zip_path).lower()
        if 'standing' in name:
            return 'standing'
        elif 'walking' in name:
            return 'walking'
        elif 'jumping' in name:
            return 'jumping'
        elif 'still' in name or 'rest' in name or 'phone_at_rest' in name:
            return 'still'
    
    return default_label


def load_sensor_data(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        accel = pd.read_csv(z.open('Accelerometer.csv'))
        gyro = pd.read_csv(z.open('Gyroscope.csv'))
    return accel, gyro


def load_all_data(data_dirs, default_labels=None):
    records = []
    for dir_path, default_label in zip(data_dirs, default_labels or [None]*len(data_dirs)):
        member = os.path.basename(dir_path)
        for fname in sorted(os.listdir(dir_path)):
            if not fname.endswith('.zip'):
                continue
            zip_path = os.path.join(dir_path, fname)
            label = get_activity_label(zip_path, default_label)
            if label is None:
                logger.warning(f'skipping {fname}, no label found')
                continue
            accel, gyro = load_sensor_data(zip_path)
            records.append({
                'file': fname,
                'member': member,
                'activity': label,
                'accel': accel,
                'gyro': gyro,
                'n_samples_accel': len(accel),
                'n_samples_gyro': len(gyro),
                'duration_s': accel['seconds_elapsed'].iloc[-1] - accel['seconds_elapsed'].iloc[0]
            })
            logger.info(f'loaded {fname} | {member} | {label} | {len(accel)} accel samples')
    return records

In [ ]:
david_tagged = 'data/david'
queen_dir = 'data/queen'

records = load_all_data(
    [david_tagged, queen_dir],
    default_labels=['still', None]
)

summary = pd.DataFrame([{
    'file': r['file'], 'member': r['member'], 'activity': r['activity'],
    'accel_samples': r['n_samples_accel'], 'duration_s': round(r['duration_s'], 2)
} for r in records])

print(f"total recordings: {len(records)}")
print(summary.groupby(['member', 'activity']).size().unstack(fill_value=0))
print(f"\ntotal duration per activity:")
print(summary.groupby('activity')['duration_s'].sum())

## 2. Data Visualization

Plot sample accelerometer and gyroscope signals for each activity to understand the data patterns.

In [ ]:
activities = ['standing', 'walking', 'jumping', 'still']

fig, axes = plt.subplots(4, 2, figsize=(14, 12))
fig.suptitle('Sample Sensor Signals per Activity', fontsize=14)

for i, activity in enumerate(activities):
    sample = [r for r in records if r['activity'] == activity][0]
    accel = sample['accel']
    gyro = sample['gyro']
    t = accel['seconds_elapsed']
    
    axes[i, 0].plot(t, accel['x'], label='x', alpha=0.8)
    axes[i, 0].plot(t, accel['y'], label='y', alpha=0.8)
    axes[i, 0].plot(t, accel['z'], label='z', alpha=0.8)
    axes[i, 0].set_title(f'{activity} - accelerometer')
    axes[i, 0].set_ylabel('m/s^2')
    axes[i, 0].legend(loc='upper right', fontsize=8)
    
    t_g = gyro['seconds_elapsed']
    axes[i, 1].plot(t_g, gyro['x'], label='x', alpha=0.8)
    axes[i, 1].plot(t_g, gyro['y'], label='y', alpha=0.8)
    axes[i, 1].plot(t_g, gyro['z'], label='z', alpha=0.8)
    axes[i, 1].set_title(f'{activity} - gyroscope')
    axes[i, 1].set_ylabel('rad/s')
    axes[i, 1].legend(loc='upper right', fontsize=8)

axes[-1, 0].set_xlabel('time (s)')
axes[-1, 1].set_xlabel('time (s)')
plt.tight_layout()
plt.savefig('logs/raw_signals.png', dpi=150)
plt.show()

## 3. Windowing and Feature Extraction

At 100Hz, we use a 1-second window (100 samples) with 50% overlap. This gives us enough data per window to capture one full gait cycle for walking (~1s) and multiple jump cycles, while keeping the window short enough to represent a single activity.

Features extracted per window:

**Time-domain:**
- Mean (per axis) — captures the average orientation/movement
- Variance (per axis) — captures intensity of movement
- RMS (root mean square of magnitude) — overall signal energy
- SMA (signal magnitude area) — total acceleration across all axes
- Correlation between axes — captures directional relationships

**Frequency-domain:**
- Dominant frequency from FFT — captures repetitive motion patterns
- Spectral energy — total power in the frequency spectrum

In [ ]:
SAMPLING_RATE = 100
WINDOW_SIZE = 100
OVERLAP = 50

def compute_features(accel_window, gyro_window, sampling_rate=SAMPLING_RATE):
    ax, ay, az = accel_window['x'].values, accel_window['y'].values, accel_window['z'].values
    gx, gy, gz = gyro_window['x'].values, gyro_window['y'].values, gyro_window['z'].values
    
    accel_mag = np.sqrt(ax**2 + ay**2 + az**2)
    gyro_mag = np.sqrt(gx**2 + gy**2 + gz**2)
    
    features = {}
    
    features['accel_mean_x'] = np.mean(ax)
    features['accel_mean_y'] = np.mean(ay)
    features['accel_mean_z'] = np.mean(az)
    
    features['accel_var_x'] = np.var(ax)
    features['accel_var_y'] = np.var(ay)
    features['accel_var_z'] = np.var(az)
    
    features['accel_rms'] = np.sqrt(np.mean(accel_mag**2))
    
    features['accel_sma'] = np.sum(np.abs(ax) + np.abs(ay) + np.abs(az)) / len(ax)
    
    if np.std(ax) > 0 and np.std(ay) > 0:
        features['accel_corr_xy'] = np.corrcoef(ax, ay)[0, 1]
    else:
        features['accel_corr_xy'] = 0.0
    if np.std(ax) > 0 and np.std(az) > 0:
        features['accel_corr_xz'] = np.corrcoef(ax, az)[0, 1]
    else:
        features['accel_corr_xz'] = 0.0
    
    features['gyro_mean_x'] = np.mean(gx)
    features['gyro_mean_y'] = np.mean(gy)
    features['gyro_mean_z'] = np.mean(gz)
    
    features['gyro_var_x'] = np.var(gx)
    features['gyro_var_y'] = np.var(gy)
    features['gyro_var_z'] = np.var(gz)
    
    n = len(accel_mag)
    freqs = fftfreq(n, d=1.0/sampling_rate)[:n//2]
    
    accel_fft = np.abs(fft(accel_mag))[:n//2]
    features['accel_dominant_freq'] = freqs[np.argmax(accel_fft[1:]) + 1]
    features['accel_spectral_energy'] = np.sum(accel_fft**2) / n
    
    gyro_fft = np.abs(fft(gyro_mag))[:n//2]
    features['gyro_dominant_freq'] = freqs[np.argmax(gyro_fft[1:]) + 1]
    features['gyro_spectral_energy'] = np.sum(gyro_fft**2) / n
    
    return features


def extract_windows(accel, gyro, window_size=WINDOW_SIZE, overlap=OVERLAP):
    windows = []
    step = window_size - overlap
    min_len = min(len(accel), len(gyro))
    for start in range(0, min_len - window_size + 1, step):
        end = start + window_size
        windows.append((accel.iloc[start:end], gyro.iloc[start:end]))
    return windows


def extract_features(records):
    all_features = []
    all_labels = []
    all_meta = []
    
    for rec in records:
        windows = extract_windows(rec['accel'], rec['gyro'])
        for w_idx, (accel_w, gyro_w) in enumerate(windows):
            feats = compute_features(accel_w, gyro_w)
            all_features.append(feats)
            all_labels.append(rec['activity'])
            all_meta.append({'file': rec['file'], 'member': rec['member'], 'window': w_idx})
    
    feature_df = pd.DataFrame(all_features)
    labels = np.array(all_labels)
    
    logger.info(f'extracted {len(feature_df)} windows with {len(feature_df.columns)} features')
    return feature_df, labels, all_meta

In [ ]:
feature_df, labels, meta = extract_features(records)

print(f"total windows: {len(feature_df)}")
print(f"features per window: {len(feature_df.columns)}")
print(f"\nwindows per activity:")
for act in activities:
    print(f"  {act}: {np.sum(labels == act)}")

feature_df.head()

## 4. Normalization

Z-score normalization is applied to ensure all features are on the same scale. This is important because features like spectral energy can be orders of magnitude larger than correlation values, and the HMM emission probabilities would be dominated by high-magnitude features otherwise.

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(feature_df.values)
feature_names = feature_df.columns.tolist()

label_map = {act: i for i, act in enumerate(activities)}
y = np.array([label_map[l] for l in labels])

print(f"feature matrix shape: {X.shape}")
print(f"labels shape: {y.shape}")
print(f"label mapping: {label_map}")

## 5. Feature Distribution Visualization

In [ ]:
key_features = ['accel_rms', 'accel_sma', 'accel_var_x', 'accel_dominant_freq',
                 'gyro_var_x', 'gyro_spectral_energy']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Feature Distributions by Activity', fontsize=14)

for idx, feat in enumerate(key_features):
    ax = axes[idx // 3, idx % 3]
    for act in activities:
        mask = labels == act
        ax.hist(feature_df[feat][mask], bins=20, alpha=0.5, label=act, density=True)
    ax.set_title(feat)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('logs/feature_distributions.png', dpi=150)
plt.show()

## 6. Baum-Welch Training

Train the HMM using the Baum-Welch (EM) algorithm with a Gaussian emission model. Convergence is checked by monitoring the change in log-likelihood between iterations, stopping when the improvement drops below a threshold (epsilon).

In [ ]:
from hmmlearn.hmm import GaussianHMM

n_states = 4
n_features = X.shape[1]

model = GaussianHMM(
    n_components=n_states,
    covariance_type='full',
    n_iter=200,
    tol=1e-4,
    random_state=42,
    verbose=True
)

model.fit(X)

training_history = {
    'converged': model.monitor_.converged,
    'n_iter': model.monitor_.iter,
    'log_likelihood': model.score(X)
}

print(f"converged: {training_history['converged']}")
print(f"iterations: {training_history['n_iter']}")
print(f"log-likelihood: {training_history['log_likelihood']:.2f}")

with open('logs/training_history.json', 'w') as f:
    json.dump(training_history, f, indent=2)

logger.info(f"training complete: {training_history}")

In [ ]:
print("learned transition matrix:")
print(np.round(model.transmat_, 3))
print(f"\ninitial state probabilities: {np.round(model.startprob_, 3)}")

## 7. Save Outputs for Part 2 (Evaluation)

Save the trained model, feature matrix, labels, scaler, and metadata so Queen can load them directly for Viterbi decoding and evaluation.

In [ ]:
import pickle

os.makedirs('outputs', exist_ok=True)

np.savez('outputs/features.npz', X=X, y=y, feature_names=feature_names)

with open('outputs/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('outputs/model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('outputs/label_map.json', 'w') as f:
    json.dump(label_map, f)

with open('outputs/meta.json', 'w') as f:
    json.dump(meta, f)

print("saved to outputs/:")
for fname in os.listdir('outputs'):
    fsize = os.path.getsize(f'outputs/{fname}')
    print(f"  {fname} ({fsize} bytes)")